# Phase 0 — Environment

This notebook validates infrastructure only. It does not download models or datasets, run ASR, or train anything.

In [ ]:
from pathlib import Path
import sys
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(ROOT / 'src'))
from streaming_asr.audio.stream import AudioStream
from streaming_asr.models.vad import DummyVAD
from streaming_asr.models.overlap_detector import DummyOverlapDetector
from streaming_asr.pipeline.streaming import StreamingASRPipeline
from streaming_asr.utils.config import Config
config = Config.from_yaml(ROOT / 'configs/base.yaml')
sample_rate = config.get('audio.sample_rate')
chunk_size = int(sample_rate * config.get('audio.chunk_ms') / 1000)
signal = np.sin(np.linspace(0, 10, chunk_size * 2)).astype(np.float32)
pipeline = StreamingASRPipeline(DummyVAD(), DummyOverlapDetector())
pipeline.start()
for chunk in AudioStream.from_array(signal, sample_rate, chunk_size):
    pipeline.process(chunk)
state = pipeline.finalize()
print({'phase': 'phase0', 'chunks': state.chunk_index + 1, 'finalized': state.finalized, 'outputs': config.get('paths.outputs')})